In [39]:
import os
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from uuid import uuid4
from PIL import Image
load_dotenv()

True

In [6]:
model_id = "openai/clip-vit-base-patch32"

In [19]:
BASE_DIR = Path().resolve().parent
image_root = BASE_DIR/"images"
MODEL_ID = "immich-app/ViT-B-32__laion2b-s34b-b79k"
print(image_root)

F:\Agentic AI\Image-Semantic-Search\images


In [13]:
from langchain_experimental.open_clip import OpenCLIPEmbeddings
embedder = OpenCLIPEmbeddings(
    model_name="ViT-B-32" , 
    checkpoint= "laion2b_s34b_b79k",
    device = "cpu"
    )

f:\Agentic AI\Image-Semantic-Search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
str(image_root/"animals"/"cat.jpeg")

'F:\\Agentic AI\\Image-Semantic-Search\\images\\animals\\cat.jpeg'

In [21]:
img_embed = embedder.embed_image([Path(str(image_root/"animal"/"cat.jpeg"))])

In [24]:
len(img_embed[0])

512

In [26]:
qdrant_api = os.getenv("QDRANT_API_KEY")
qdrant_endpoint = os.getenv("API_ENDPOINT")

In [32]:
from qdrant_client import QdrantClient
from qdrant_client.http import models 
client = QdrantClient(url=qdrant_endpoint , api_key= qdrant_api)

In [33]:
collections = client.get_collections().collections
collections

[]

In [31]:
COLLECTION_NAME = "semantic_image_search"
VECTOR_SIZE = 512

In [36]:
collections = client.get_collections().collections
existing_names = {c.name for c in collections}
existing_names

{'semantic_image_search'}

In [37]:
if COLLECTION_NAME not in existing_names:
    print(f"Creating Collection : {COLLECTION_NAME}")
    client.create_collection(collection_name = COLLECTION_NAME  , vectors_config= models.VectorParams(size=VECTOR_SIZE , distance= models.Distance.COSINE))
else:
    print(f"Collection Already Exist : {COLLECTION_NAME}")

Collection Already Exist : semantic_image_search


In [40]:
def index_image(image_path , category=None):
    image_embed = embedder.embed_image([str(image_path)])[0]
    emb = np.array(image_embed).tolist()
    payload = {"filename" : os.path.basename(image_path) , "path" : image_path , "category" : category}
    client.upsert(
        collection_name= COLLECTION_NAME , 
        points = [
            models.PointStruct(
                id = str(uuid4()) , 
                vector= emb , 
                payload= payload)])
    print(f"Indexed : {image_path}")

In [41]:
cat_image_path = image_root/"animal"/"cat.jpeg"
index_image(cat_image_path , category="animal")

Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\cat.jpeg


In [ ]:
def index_folder(root_folder):
    exts = (".jpg", ".jpeg", ".png", ".webp")
    for dirpath, _, files in os.walk(root_folder):
        category = os.path.basename(dirpath)
        for f in files:
            if f.lower().endswith(exts):
                img_path = os.path.join(dirpath, f)
                # print(img_path,category)
                index_image(img_path,category=category)

In [43]:
index_folder(image_root)

Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\cat.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\crocodile.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\crocodile_1.png
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\dog.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\elephant.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\giraffe.webp
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\horse.webp
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\lion.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\panda.jpg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\tiger.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\animal\zebra.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lavender.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lily.jpeg
Indexed : F:\Agentic AI\Image-Semantic-Search\images\flower\lotus.j